In [4]:
from typing import Tuple
import torch
from torch import Tensor
import torch.nn as nn
import math


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

def min_zero_row(zero_mat: Tensor) -> Tuple[Tensor, Tensor]:
    sum_zero_mat = zero_mat.sum(1)
    sum_zero_mat[sum_zero_mat == 0] = 9999

    zero_row = sum_zero_mat.min(0)[1]
    zero_column = zero_mat[zero_row].nonzero()[0]

    zero_mat[zero_row, :] = False
    zero_mat[:, zero_column] = False

    mark_zero = torch.tensor([[zero_row, zero_column]], device = device)
    return zero_mat, mark_zero

def mark_matrix(mat: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
    zero_bool_mat = (mat == 0)
    zero_bool_mat_copy = zero_bool_mat.clone()

    marked_zero = torch.tensor([], device = device)
    while (True in zero_bool_mat_copy):
        zero_bool_mat_copy, mark_zero = min_zero_row(zero_bool_mat_copy)
        marked_zero = torch.concat([marked_zero, mark_zero], dim = 0)

    marked_zero_row = marked_zero[:, 0]
    marked_zero_col = marked_zero[:, 1]

    arange_index_row = torch.arange(mat.shape[0], dtype=torch.float, device = device).unsqueeze(1)
    
    repeated_marked_row = marked_zero_row.repeat(mat.shape[0], 1)
    bool_non_marked_row = torch.all(arange_index_row != repeated_marked_row, dim = 1)
    non_marked_row = arange_index_row[bool_non_marked_row].squeeze()

    non_marked_mat = zero_bool_mat[non_marked_row.long(), :]
    marked_cols = non_marked_mat.nonzero().unique()

    is_need_add_row = True
    while is_need_add_row:
        repeated_non_marked_row = non_marked_row.repeat(marked_zero_row.shape[0], 1)
        repeated_marked_cols = marked_cols.repeat(marked_zero_col.shape[0], 1)

        first_bool = torch.all(marked_zero_row.unsqueeze(1) != repeated_non_marked_row, dim = 1)
        second_bool = torch.any(marked_zero_col.unsqueeze(1) == repeated_marked_cols, dim = 1)

        addit_non_marked_row = marked_zero_row[first_bool & second_bool]

        if addit_non_marked_row.shape[0] > 0:
            non_marked_row = torch.concat([non_marked_row.reshape(-1), addit_non_marked_row[0].reshape(-1)])
        else:
            is_need_add_row = False

    repeated_non_marked_row = non_marked_row.repeat(mat.shape[0], 1)
    bool_marked_row = torch.all(arange_index_row != repeated_non_marked_row, dim = 1)
    marked_rows = arange_index_row[bool_marked_row].squeeze(0)

    return marked_zero, marked_rows, marked_cols

def adjust_matrix(mat: Tensor, cover_rows: Tensor, cover_cols: Tensor) -> Tensor:
    bool_cover = torch.zeros_like(mat)
    bool_cover[cover_rows.long()] = True
    bool_cover[:, cover_cols.long()] = True

    non_cover = mat[bool_cover != True]
    min_non_cover = non_cover.min()

    mat[bool_cover != True] = mat[bool_cover != True] - min_non_cover

    double_bool_cover = torch.zeros_like(mat)
    double_bool_cover[cover_rows.long(), cover_cols.long()] = True

    mat[double_bool_cover == True] = mat[double_bool_cover == True] + min_non_cover

    return mat

def hungarian_algorithm(mat: Tensor) -> Tensor:
    dim = mat.shape[0]
    cur_mat = mat.clone()

    cur_mat = cur_mat - cur_mat.min(1, keepdim=True)[0]
    cur_mat = cur_mat - cur_mat.min(0, keepdim=True)[0]

    zero_count = 0
    iters = 0
    while zero_count < dim and iters < 100:
        ans_pos, marked_rows, marked_cols = mark_matrix(cur_mat)
        zero_count = len(marked_rows) + len(marked_cols)

        if zero_count < dim:
            cur_mat = adjust_matrix(cur_mat, marked_rows, marked_cols)
        iters += 1

    # Create permutation matrix
    perm_matrix = torch.zeros_like(mat)
    for pos in ans_pos:
        i, j = int(pos[0]), int(pos[1])  # Ensure indices are integers
        perm_matrix[i, j] = 1

    return perm_matrix

def sinkhorn_logspace(logP, niters=10):
    for _ in range(niters):
        # Normalize columns and take the log again
        logP = logP - torch.logsumexp(logP, dim=0, keepdim=True)
        # Normalize rows and take the log again
        logP = logP - torch.logsumexp(logP, dim=1, keepdim=True)
    return logP

In [5]:
# Test code to check ELBO calculation
# Initialize parameters
n_locations = 5
batch_size = 50
n_sample = 100

# Create dummy data
torch.manual_seed(42)  # Set seed for reproducibility
X = torch.randn(batch_size, n_locations)
Y = 0.5 * X + 0.1 * torch.randn(batch_size, n_locations)
mu_lambda_beta = 0.5
sigmasq_lambda_beta = 0.1
M_S_star = torch.eye(n_locations)
mu_W = torch.randn(batch_size, n_locations)
eta_X = 10
lambda_a2 = 1.0
lambda_b2 = 1.0
tau_X = 0.1

# Initialize the model
MX = 1/n_locations * torch.rand(n_locations, n_locations)
VX = 0.2 * torch.rand(n_locations, n_locations)

In [26]:
class vi_piX(nn.Module):
    def __init__(self, n_locations):
        super(vi_piX, self).__init__()
        self.n_locations = n_locations
        self.MX = nn.Parameter(torch.log(1/torch.tensor(n_locations)) * torch.ones(n_locations, n_locations, requires_grad=True))
        self.VX = nn.Parameter(5*torch.ones(n_locations, n_locations, requires_grad=True))
        #self.VX_unconstrained = nn.Parameter(torch.full((n_locations, n_locations), 0.2))
            
    def forward(self, Y, X, mu_lambda_beta,
                sigmasq_lambda_beta, M_S_star, mu_W,
                eta_X, lambda_a2, lambda_b2, 
                tau_X = 0.1, n_sample = 100):
        
        # Enable anomaly detection
        torch.autograd.set_detect_anomaly(True)

        # sample piX
        torch.manual_seed(42)  # Set seed for reproducibility
        sampled_piX = torch.zeros(n_sample, self.n_locations, self.n_locations)

        # calculate nearest doubly stochastic matrix to MX once
        log_MX = self.MX
        log_MX_tilde = sinkhorn_logspace(log_MX, niters=10)
        MX_tilde = torch.exp(log_MX_tilde)
        #VX = torch.nn.functional.softplus(self.VX_unconstrained)
    

        # Compute the ELBO
        B = Y.shape[0]
        elbo = 0.0
        for i in range(n_sample):
            Phi = MX_tilde + torch.sqrt(torch.exp(self.VX)) * torch.randn(self.n_locations, self.n_locations)
            # sampled_piX[i] = tau_X * Phi + (1 - tau_X) * hungarian_algorithm(-Phi.clone())
            # current_piX = sampled_piX[i] # Access the current sample of piX
            current_piX = tau_X * Phi + (1 - tau_X) * hungarian_algorithm(-Phi.clone())

            # Vectorized computation for all regions
            term1 = 0.
            for j in range(B):
                Y_i = Y[j]                    # (d,)
                X_i = X[j]                    # (d,)
                mu_Wi = mu_W[j]              # (d,)

                # (1) 2 * Y_i^T * pi_x * X_i * mu
                part1 = 2 * mu_lambda_beta * torch.dot(Y_i, current_piX @ X_i)

                # (2) (mu^2 + sigma^2) * X_i^T * pi_x^T * pi_x * X_i
                temp = current_piX @ X_i            # (d,)
                part2 = (mu_lambda_beta ** 2 + sigmasq_lambda_beta) * temp.dot(temp)

                # (3) 2 * mu * X_i^T * pi_x^T * M_star_S * mu_Wi
                part3 = 2 * mu_lambda_beta * (temp.T @ M_S_star @ mu_Wi)

                term1 = term1 + part1 + part2 + part3

            coeff = -lambda_a2 / (2 * lambda_b2)
            total_term1 = coeff * term1

            # Second summation: over entries of pi_x
            x_mk_squared = current_piX.pow(2)
            x_mk_minus1_squared = (current_piX - 1).pow(2)

            exponent1 = -x_mk_squared / (2 * eta_X**2)
            exponent2 = -x_mk_minus1_squared / (2 * eta_X**2)

            # Stable log-sum-exp
            log_term = -0.5 * torch.logsumexp(torch.stack([exponent1, exponent2]), dim=0)
            total_log_term = torch.sum(log_term)

            # Final terms
            neg_log_tauX = self.n_locations ** 2 * torch.log(torch.tensor(tau_X))

            # Update ELBO
            elbo += total_term1 + total_log_term + neg_log_tauX + 2 * torch.sum(torch.log(self.VX))
            
            #elbo += total_term1 + neg_log_tauX + 2 * self.VX.sum()

        
        elbo = elbo / n_sample 
        return -elbo
    
    

import torch.optim as optim

# Initialize the model
model = vi_piX(n_locations=n_locations)

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Number of optimization steps
n_steps = 1000

# Minimize the model
for step in range(n_steps):
    optimizer.zero_grad()  # Clear gradients
    loss = model(Y, X, mu_lambda_beta, sigmasq_lambda_beta, M_S_star, mu_W, eta_X, lambda_a2, lambda_b2, tau_X, n_sample)
    print(loss)
    #torch.autograd.set_detect_anomaly(True)  # Enable anomaly detection
    loss.backward()  # Compute gradients
    optimizer.step()  # Update parameters

    # Print loss every 100 steps
    if step % 1 == 0:
        print(f"Step {step}, Loss: {loss.item()}")
        print("Current MX:")
        print(torch.exp(model.MX))

        print("\nCurrent VX:")
        print(torch.exp(model.VX))

tensor(440.0875, grad_fn=<NegBackward0>)
Step 0, Loss: 440.0874938964844
Current MX:
tensor([[0.1980, 0.1980, 0.2020, 0.2020, 0.1980],
        [0.2020, 0.1980, 0.1980, 0.1980, 0.2020],
        [0.1980, 0.2020, 0.1980, 0.1980, 0.2020],
        [0.2020, 0.2020, 0.1980, 0.1980, 0.2020],
        [0.1980, 0.2020, 0.2020, 0.2020, 0.1980]], grad_fn=<ExpBackward0>)

Current VX:
tensor([[146.9364, 146.9364, 146.9364, 146.9364, 146.9364],
        [146.9364, 146.9364, 146.9364, 146.9364, 146.9364],
        [146.9364, 146.9364, 146.9364, 146.9364, 146.9364],
        [146.9364, 146.9364, 146.9364, 146.9364, 146.9364],
        [146.9364, 146.9364, 146.9364, 146.9364, 146.9364]],
       grad_fn=<ExpBackward0>)
tensor(436.5389, grad_fn=<NegBackward0>)
Step 1, Loss: 436.5389404296875
Current MX:
tensor([[0.1960, 0.1960, 0.2040, 0.2040, 0.1960],
        [0.2040, 0.1960, 0.1960, 0.1960, 0.2040],
        [0.1960, 0.2040, 0.1960, 0.1960, 0.2040],
        [0.2040, 0.2040, 0.1960, 0.1960, 0.2040],
        [0

KeyboardInterrupt: 